In [0]:
# configuração inicial
CATALOG = "workspace"

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

print(f"Catalog: {CATALOG}")
print(f"Bronze: {CATALOG}.{BRONZE_SCHEMA}")
print(f"Silver: {CATALOG}.{SILVER_SCHEMA}")

Catalog: workspace
Bronze: workspace.bronze
Silver: workspace.silver


In [0]:
# imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# leitura da bronze
SOURCE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.tb_movies_info"
TARGET_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.tb_info_filmes"

df_movies_info = spark.table(SOURCE_TABLE)

print(f"Tabela origem: {SOURCE_TABLE}")
print(f"Quantidade: {df_movies_info.count()}")

df_movies_info.printSchema()

Tabela origem: workspace.bronze.tb_movies_info
Quantidade: 106930
root
 |-- id: string (nullable = true)
 |-- tconst: string (nullable = true)
 |-- title: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- runtime: string (nullable = true)
 |-- status: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:
# renomeação de colunas
df_info = df_movies_info.select(
    F.col("id").alias("id_filme"),
    F.col("title").alias("titulo"),
    F.col("original_title").alias("titulo_original"),
    F.col("release_date").alias("data_lancamento"),
    F.col("runtime").alias("duracao_minutos"),
    F.col("original_language").alias("idioma_original"),
    F.col("status").alias("status_filme"),
    F.col("overview").alias("sinopse"),
    F.col("tagline").alias("frase_divulgacao"),
    F.col("ingestion_datetime")
)

display(df_info.limit(20))

id_filme,titulo,titulo_original,data_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao,ingestion_datetime
293660,Deadpool,Deadpool,2016-02-09,108,en,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-09-19T20:10:34.961Z
299536,AVENGERS: INFINITY WAR,Avengers: Infinity War,04-25-2018,149,en,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-09-19T20:10:34.961Z
299534,Avengers: Endgame,Avengers: Endgame,2019-04-24,181,en,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-09-19T20:10:34.961Z
475557,Joker,Joker,2019-10-01,122,en,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-09-19T20:10:34.961Z
271110,Captain America: Civil War,Captain America: Civil War,2016-04-27,147,en,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-09-19T20:10:34.961Z
284054,Black Panther,Black Panther,2018-02-13,135,en,Released,"King T'Challa returns home to the reclusive, technologically advanced African nation of Wakanda to serve as his country's new leader. However, T'Challa soon finds that he is challenged for the throne by factions within his own country as well as without. Using powers reserved to Wakandan kings, T'Challa assumes the Black Panther mantle to join with ex-girlfriend Nakia, the queen-mother, his princess-kid sister, members of the Dora Milaje (the Wakandan 'special forces') and an American secret agent, to prevent Wakanda from being dragged into a world war.",null,2026-09-19T20:10:34.961Z
284052,Doctor Strange,Doctor Strange,2016-10-25,115,en,Released,"After his career is destroyed, a brilliant but arrogant surgeon gets a new lease on life when a sorcerer takes him under her wing and trains him to defend the world against evil.",The impossibilities are endless.,2026-09-19T20:10:34.961Z
315635,Spider-Man: Homecoming,Spider-Man: Homecoming,2017-07-05,133,en,RELEASED,"Following the events of Captain America: Civil War, Peter Parker, with the help of his mentor Tony Stark, tries to balance his life as an ordinary high school student in Queens, New York City, with fighting crime as his superhero alter ego Spider-Man as a new threat, the Vulture, emerges.",Homework can wait. The city can't.,2026-09-19T20:10:34.961Z
283995,Guardians of the Galaxy Vol. 2,Guardians of the Galaxy Vol. 2,2017-04-19,137,en,Released,The Guardians must fight to keep their newfound family together as they unravel the mysteries of Peter Quill's true parentage.,Obvious

In [0]:
# investigação dos status
display(
    df_info
    .groupBy("status_filme")
    .count()
    .orderBy(F.desc("count"))
)

status_filme,count
Released,80265
RELEASED,16031
released,9107
Post Production,531
In Production,460
IN PRODUCTION,105
POST PRODUCTION,93
post production,74
null,66
in production,55


In [0]:
# normalizar status
df_info = df_info.withColumn(
    "status_normalizado",
    F.lower(
        F.trim(
            F.regexp_replace(
                F.col("status_filme"),
                r"[-\s]+",
                " "
            )
        )
    )
)

# verificar se a normalização está correta
display(
    df_info.select(
        "status_filme",
        "status_normalizado"
    ).distinct()
)

status_filme,status_normalizado
Released,released
released,released
RELEASED,released
null,null
his errant dad returns,his errant dad returns
sometimes referred to the Islamic Religious Police. All this,sometimes referred to the islamic religious police. all this
"\Can you still open your heart to a friend who turns out to be """"\""""Boy A""""","\can you still open your heart to a friend who turns out to be """"\""""boy a"""""
In Production,in production
IN PRODUCTION,in production
Post Production,post production


In [0]:
df_info = df_info.withColumn(
    "status_filme",

    F.when(
        F.col("status_normalizado") == "released",
        "Lançado"
    )
    .when(
        F.col("status_normalizado") == "post production",
        "Pós-Produção"
    )
    .when(
        F.col("status_normalizado") == "in production",
        "Em Produção"
    )
    .when(
        F.col("status_normalizado") == "planned",
        "Planejado"
    )
    .when(
        F.col("status_normalizado") == "rumored",
        "Rumores"
    )
    .when(
        F.col("status_normalizado") == "canceled",
        "Cancelado"
    )
    .otherwise("Não Informado")
)

df_info = df_info.drop("status_normalizado")

display(
    df_info
    .groupBy("status_filme")
    .count()
    .orderBy(F.desc("count"))
)

status_filme,count
Lançado,105403
Pós-Produção,743
Em Produção,666
Não Informado,70
Planejado,48


In [0]:
# visualização da data de lançamento
display(
    df_info
    .select("data_lancamento")
    .where(F.col("data_lancamento").isNotNull())
    .distinct()
    .limit(100)
)

data_lancamento
2016-02-09
04-25-2018
2019-04-24
2019-10-01
2016-04-27
2018-02-13
2016-10-25
2017-07-05
2017-04-19
2016-08-03


In [0]:
# visualização da duração
display(
    df_info
    .select("duracao_minutos")
    .where(F.col("duracao_minutos").isNotNull())
    .distinct()
    .limit(100)
)

duracao_minutos
108
149
181
122
147
135
115
133
137
123


In [0]:
# conversão de 'data_lancamento'
df_info = df_info.withColumn(
    "data_lancamento",
    F.coalesce(
        F.expr(
            "try_to_timestamp(data_lancamento, 'yyyy-MM-dd')"
        ),
        F.expr(
            "try_to_timestamp(data_lancamento, 'MM-dd-yyyy')"
        ),
        F.expr(
            "try_to_timestamp(data_lancamento, 'dd/MM/yyyy')"
        )
    ).cast("date")
)

In [0]:
# conversão de 'duracao_minutos'
df_info = df_info.withColumn(
    "duracao_minutos",
    F.expr("try_cast(duracao_minutos as int)")
)

In [0]:
# conversão de 'id_filme
df_info = df_info.withColumn(
    "id_filme",
    F.col("id_filme").cast("string")
)

In [0]:
# deduplicação
window_filme = (
    Window
    .partitionBy("id_filme")
    .orderBy(F.col("ingestion_datetime").desc())
)

df_info = (
    df_info
    .withColumn(
        "_row_number",
        F.row_number().over(window_filme)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

In [0]:
# criar ano_lancamento
df_info = df_info.withColumn(
    "ano_lancamento",
    F.year(F.col("data_lancamento"))
)

In [0]:
# organizar estrutura final da camada silver (remoção do ingestion_datetime)
df_info_final = df_info.select(
    "id_filme",
    "titulo",
    "titulo_original",
    "data_lancamento",
    "ano_lancamento",
    "duracao_minutos",
    "idioma_original",
    "status_filme",
    "sinopse",
    "frase_divulgacao"
)

df_info_final.printSchema()

display(df_info_final.limit(20))

root
 |-- id_filme: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- titulo_original: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- ano_lancamento: integer (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = false)
 |-- sinopse: string (nullable = true)
 |-- frase_divulgacao: string (nullable = true)



id_filme,titulo,titulo_original,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao
1000004,Purple Beatz,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry.",A drum n bass romance.
1000005,Aisha Brown: The First Black Woman Ever,Aisha Brown: The First Black Woman Ever,2020-02-14,2020,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs.",null
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,Kyle Brownrigg: Introducing Lyle,2022-05-27,2022,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle.",null
1000011,Worth Your Weight in Gold,O Teu Peso Em Ouro,2022-07-14,2022,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness.",null
1000014,On va manquer !,On va manquer !,2018-05-15,2018,0,fr,Lançado,null,null
1000030,58 Hours: The Baby Jessica Story,58 Hours: The Baby Jessica Story,2021-07-31,2021,0,es,Lançado,null,null
1000054,One Hundred Years and Hope,百年と希望,2022-06-18,2022,107,ja,Lançado,"In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope.",null
1000058,Homecoming,Le retour,2023-07-12,2023,110,fr,Lançado,"Kheìdidja, in her forties, works for a wealthy Parisian family who offers her the opportunity to take care of their children for a summer in Corsica. It's an opportunity for her to return with her daughters, Jessica and Farah, to the island they left fifteen years earlier in tragic circumstances.",null
1000059,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,2016-04-05,2016,116,ja,Lançado,"Edawakare, the driver of the Time Taxi that allows passengers to return to their life's turning points, visits a hot spring this time around. An array of guests at the inn are fraught with troubles in their life and become passengers of the Time Taxi. And somehow all the clients are actually part of a bigger picture?!",null
1000073,A Chance To Win,Pour l'honneur,2023-05-03,2023,97,fr,Lançado,"Two villages in the south of France have always been bitter rivals, but when a group of asylum seekers arrive in the community, the life of both villages is shaken up and age-old disagreements escalate. Their antagonism reaches its peak with the annual rugby derby played between the two village teams, but this time, with the new outsiders joining as unexpected recruits, the result of the 100th match will be more unpredictable than ever.",null


In [0]:
# validação antes de salvar
total_registros = df_info_final.count()

total_ids_distintos = (
    df_info_final
    .select("id_filme")
    .distinct()
    .count()
)

ids_nulos = (
    df_info_final
    .filter(F.col("id_filme").isNull())
    .count()
)

print("VALIDAÇÃO — TB_INFO_FILMES")

print(f"Total de registros:       {total_registros}")
print(f"IDs distintos:            {total_ids_distintos}")
print(f"IDs nulos:                {ids_nulos}")

if total_registros == total_ids_distintos:
    print("[OK] Não existem IDs duplicados.")
else:
    print("[ERRO] Existem IDs duplicados.")

VALIDAÇÃO — TB_INFO_FILMES
Total de registros:       97879
IDs distintos:            97879
IDs nulos:                0
[OK] Não existem IDs duplicados.


In [0]:
# verificar datas que viraram NULL
datas_nulas = (
    df_info_final
    .filter(F.col("data_lancamento").isNull())
    .count()
)

print(
    f"Registros com data_lancamento NULL: "
    f"{datas_nulas}"
)

Registros com data_lancamento NULL: 64


In [0]:
# validar status
display(
    df_info_final
    .groupBy("status_filme")
    .count()
    .orderBy(F.desc("count"))
)

status_filme,count
Lançado,96463
Pós-Produção,701
Em Produção,604
Não Informado,64
Planejado,47


In [0]:
(
    df_info_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(
    f"[OK] Tabela criada/atualizada: "
    f"{TARGET_TABLE}"
)

[OK] Tabela criada/atualizada: workspace.silver.tb_info_filmes


In [0]:
df_validacao = spark.table(TARGET_TABLE)

print("VALIDAÇÃO FINAL — SILVER.TB_INFO_FILMES")

print(f"Tabela: {TARGET_TABLE}")
print(f"Registros: {df_validacao.count()}")

df_validacao.printSchema()

display(df_validacao.limit(20))

VALIDAÇÃO FINAL — SILVER.TB_INFO_FILMES
Tabela: workspace.silver.tb_info_filmes
Registros: 97879
root
 |-- id_filme: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- titulo_original: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- ano_lancamento: integer (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = true)
 |-- sinopse: string (nullable = true)
 |-- frase_divulgacao: string (nullable = true)



id_filme,titulo,titulo_original,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao
1000004,Purple Beatz,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry.",A drum n bass romance.
1000005,Aisha Brown: The First Black Woman Ever,Aisha Brown: The First Black Woman Ever,2020-02-14,2020,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs.",null
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,Kyle Brownrigg: Introducing Lyle,2022-05-27,2022,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle.",null
1000011,Worth Your Weight in Gold,O Teu Peso Em Ouro,2022-07-14,2022,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness.",null
1000014,On va manquer !,On va manquer !,2018-05-15,2018,0,fr,Lançado,null,null
1000030,58 Hours: The Baby Jessica Story,58 Hours: The Baby Jessica Story,2021-07-31,2021,0,es,Lançado,null,null
1000054,One Hundred Years and Hope,百年と希望,2022-06-18,2022,107,ja,Lançado,"In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope.",null
1000058,Homecoming,Le retour,2023-07-12,2023,110,fr,Lançado,"Kheìdidja, in her forties, works for a wealthy Parisian family who offers her the opportunity to take care of their children for a summer in Corsica. It's an opportunity for her to return with her daughters, Jessica and Farah, to the island they left fifteen years earlier in tragic circumstances.",null
1000059,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,2016-04-05,2016,116,ja,Lançado,"Edawakare, the driver of the Time Taxi that allows passengers to return to their life's turning points, visits a hot spring this time around. An array of guests at the inn are fraught with troubles in their life and become passengers of the Time Taxi. And somehow all the clients are actually part of a bigger picture?!",null
1000073,A Chance To Win,Pour l'honneur,2023-05-03,2023,97,fr,Lançado,"Two villages in the south of France have always been bitter rivals, but when a group of asylum seekers arrive in the community, the life of both villages is shaken up and age-old disagreements escalate. Their antagonism reaches its peak with the annual rugby derby played between the two village teams, but this time, with the new outsiders joining as unexpected recruits, the result of the 100th match will be more unpredictable than ever.",null


In [0]:
FINANCIAL_SOURCE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.tb_movies_financials"
)

FINANCIAL_TARGET = (
    f"{CATALOG}.{SILVER_SCHEMA}.tb_financeiro_filmes"
)

df_financial_raw = spark.table(FINANCIAL_SOURCE)

print(f"Origem: {FINANCIAL_SOURCE}")
print(f"Registros: {df_financial_raw.count()}")

df_financial_raw.printSchema()

display(df_financial_raw.limit(20))

Origem: workspace.bronze.tb_movies_financials
Registros: 106165
root
 |-- id: string (nullable = true)
 |-- budget: string (nullable = true)
 |-- revenue: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



id,budget,revenue,ingestion_datetime
293660,58000000,Unknown,2026-09-19T17:52:05.439Z
299536,300000000,2052415039,2026-09-19T17:52:05.439Z
299534,356000000,2800000000,2026-09-19T17:52:05.439Z
475557,55000000,1074458282,2026-09-19T17:52:05.439Z
271110,250000000,Não Informado,2026-09-19T17:52:05.439Z
284054,200000000,1349926083,2026-09-19T17:52:05.439Z
284052,180000000,676343174,2026-09-19T17:52:05.439Z
315635,175000000,880166924,2026-09-19T17:52:05.439Z
283995,200000000,863756051,2026-09-19T17:52:05.439Z
297761,175000000,746846894,2026-09-19T17:52:05.439Z


In [0]:
display(
    df_financial_raw
    .select("budget")
    .where(F.col("budget").isNotNull())
    .distinct()
    .limit(100)
)

budget
58000000
300000000
356000000
55000000
250000000
200000000
180000000
175000000
149000000
$ 97000000


In [0]:
display(
    df_financial_raw
    .select("revenue")
    .where(F.col("revenue").isNotNull())
    .distinct()
    .limit(100)
)

revenue
Unknown
2052415039
2800000000
1074458282
Não Informado
1349926083
676343174
880166924
863756051
746846894


In [0]:
display(
    df_financial_raw
    .filter(
        F.col("budget").rlike("[^0-9.,+-]")
        |
        F.col("revenue").rlike("[^0-9.,+-]")
    )
    .select(
        "id",
        "budget",
        "revenue"
    )
    .limit(100)
)

id,budget,revenue
293660,58000000,Unknown
271110,250000000,Não Informado
263115,$ 97000000,619021436
354912,$ 175000000,-800526015
209112,$ 250000000,Não Informado
383498,110000000,Unknown
374720,USD 150000000,527000000
339403,34.0M,226945087
333339,175000000,Não Informado
297802,160000000,Não Informado


In [0]:
display(
    df_financial_raw
    .filter(
        F.trim(F.col("budget")).isin(
            "0", "0.0", "0.00", "-1"
        )
        |
        F.trim(F.col("revenue")).isin(
            "0", "0.0", "0.00", "-1"
        )
    )
    .select(
        "id",
        "budget",
        "revenue"
    )
    .limit(100)
)

id,budget,revenue
372058,0,357986087
405774,19800000,0
791373,70000000,0
466282,0,0
454983,0,0
497582,$ 21000000,0
545609,$ 65000000,0
400106,90000000,0
613504,0,Não Informado
583083,0,0


In [0]:
COTACAO_SOURCE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.tb_cotacao_dolar"
)

df_cotacao_raw = spark.table(COTACAO_SOURCE)

print(f"Origem: {COTACAO_SOURCE}")
print(f"Registros: {df_cotacao_raw.count()}")

df_cotacao_raw.printSchema()

display(
    df_cotacao_raw
    .orderBy("dataHoraCotacao")
)

Origem: workspace.bronze.tb_cotacao_dolar
Registros: 5
root
 |-- cotacaoCompra: double (nullable = true)
 |-- dataHoraCotacao: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



cotacaoCompra,dataHoraCotacao,ingestion_datetime
5.169,2026-09-14 13:10:08.144425,2026-09-19T20:37:33.810Z
5.1484,2026-09-15 13:09:19.199664,2026-09-19T20:37:33.810Z
5.152,2026-09-16 13:05:30.35873,2026-09-19T20:37:33.810Z
5.1515,2026-09-17 13:03:21.858212,2026-09-19T20:37:33.810Z
5.1569,2026-09-18 13:03:34.742036,2026-09-19T20:37:33.810Z


In [0]:
df_financial = df_financial_raw.select(
    F.col("id").cast("string").alias("id_filme"),
    F.col("budget").alias("orcamento_raw"),
    F.col("revenue").alias("receita_raw"),
    F.col("ingestion_datetime")
)

display(df_financial.limit(20))

id_filme,orcamento_raw,receita_raw,ingestion_datetime
293660,58000000,Unknown,2026-09-19T17:52:05.439Z
299536,300000000,2052415039,2026-09-19T17:52:05.439Z
299534,356000000,2800000000,2026-09-19T17:52:05.439Z
475557,55000000,1074458282,2026-09-19T17:52:05.439Z
271110,250000000,Não Informado,2026-09-19T17:52:05.439Z
284054,200000000,1349926083,2026-09-19T17:52:05.439Z
284052,180000000,676343174,2026-09-19T17:52:05.439Z
315635,175000000,880166924,2026-09-19T17:52:05.439Z
283995,200000000,863756051,2026-09-19T17:52:05.439Z
297761,175000000,746846894,2026-09-19T17:52:05.439Z


In [0]:
"""
    Limpa valores monetários da base.

    Exemplos:
    58000000       -> 58000000
    $ 97000000     -> 97000000
    USD 150000000  -> 150000000
    34.0M          -> 34000000
    Unknown        -> NULL
    Não Informado  -> NULL

    Valores zero ou negativos também são convertidos para NULL.
    """
def limpar_valor_monetario(coluna):
    
    valor = F.trim(F.col(coluna))

    # Detecta valores em milhões: 34.0M, 200.0M etc.
    valor_milhoes = (
        F.regexp_extract(
            valor,
            r"(?i)([-+]?[0-9]+(?:\.[0-9]+)?)\s*M",
            1
        )
        .cast("decimal(18,2)")
    )

    # Remove símbolos/textos, preservando dígitos,
    # sinal negativo e ponto decimal.
    valor_limpo = F.regexp_replace(
        valor,
        r"[^0-9.\-]",
        ""
    )

    valor_numerico = (
        F.when(
            F.upper(valor).isin(
                "UNKNOWN",
                "NÃO INFORMADO",
                "NAO INFORMADO",
                "N/A",
                "NULL",
                ""
            ),
            F.lit(None)
        )
        .when(
            F.upper(valor).rlike(r"[-+]?[0-9]+(?:\.[0-9]+)?\s*M"),
            valor_milhoes * F.lit(1000000)
        )
        .otherwise(
            F.expr(
                f"try_cast(regexp_replace({coluna}, '[^0-9.\\\\-]', '') "
                f"AS DECIMAL(18,2))"
            )
        )
    )

    return (
        F.when(
            valor_numerico > 0,
            valor_numerico
        )
        .otherwise(F.lit(None))
        .cast("decimal(18,2)")
    )

In [0]:
df_financial = (
    df_financial
    .withColumn(
        "orcamento_usd",
        limpar_valor_monetario("orcamento_raw")
    )
    .withColumn(
        "receita_usd",
        limpar_valor_monetario("receita_raw")
    )
)

display(
    df_financial.select(
        "orcamento_raw",
        "orcamento_usd",
        "receita_raw",
        "receita_usd"
    ).limit(100)
)

orcamento_raw,orcamento_usd,receita_raw,receita_usd
58000000,58000000.00,Unknown,null
300000000,300000000.00,2052415039,2052415039.00
356000000,356000000.00,2800000000,2800000000.00
55000000,55000000.00,1074458282,1074458282.00
250000000,250000000.00,Não Informado,null
200000000,200000000.00,1349926083,1349926083.00
180000000,180000000.00,676343174,676343174.00
175000000,175000000.00,880166924,880166924.00
200000000,200000000.00,863756051,863756051.00
175000000,175000000.00,746846894,746846894.00


In [0]:
# validar valores inválidos
print("VALIDAÇÃO — LIMPEZA FINANCEIRA")

orcamentos_nulos = (
    df_financial
    .filter(F.col("orcamento_usd").isNull())
    .count()
)

receitas_nulas = (
    df_financial
    .filter(F.col("receita_usd").isNull())
    .count()
)

orcamentos_invalidos = (
    df_financial
    .filter(F.col("orcamento_usd") <= 0)
    .count()
)

receitas_invalidas = (
    df_financial
    .filter(F.col("receita_usd") <= 0)
    .count()
)

print(f"Orçamentos NULL:          {orcamentos_nulos}")
print(f"Receitas NULL:            {receitas_nulas}")
print(f"Orçamentos <= 0 restantes: {orcamentos_invalidos}")
print(f"Receitas <= 0 restantes:   {receitas_invalidas}")

VALIDAÇÃO — LIMPEZA FINANCEIRA
Orçamentos NULL:          97281
Receitas NULL:            102583
Orçamentos <= 0 restantes: 0
Receitas <= 0 restantes:   0


In [0]:
window_financial = (
    Window
    .partitionBy("id_filme")
    .orderBy(
        F.col("ingestion_datetime").desc()
    )
)

df_financial = (
    df_financial
    .withColumn(
        "_row_number",
        F.row_number().over(window_financial)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

print(
    f"Registros após deduplicação: "
    f"{df_financial.count()}"
)

print(
    f"IDs distintos: "
    f"{df_financial.select('id_filme').distinct().count()}"
)

Registros após deduplicação: 99006
IDs distintos: 99006


In [0]:
df_cotacao = (
    df_cotacao_raw
    .withColumn(
        "data_cotacao",
        F.to_date("dataHoraCotacao")
    )
    .withColumn(
        "cotacao_compra",
        F.col("cotacaoCompra").cast("decimal(18,4)")
    )
    .select(
        "data_cotacao",
        "cotacao_compra"
    )
)

display(
    df_cotacao.orderBy("data_cotacao")
)

data_cotacao,cotacao_compra
2026-09-14,5.1690
2026-09-15,5.1484
2026-09-16,5.1520
2026-09-17,5.1515
2026-09-18,5.1569


In [0]:
limites = (
    df_cotacao
    .agg(
        F.min("data_cotacao").alias("data_min"),
        F.max("data_cotacao").alias("ultima_cotacao")
    )
    .first()
)

data_min = limites["data_min"]

# A série deve continuar até o dia atual,
# mesmo que não exista cotação nesse dia.
data_max = spark.sql(
    "SELECT current_date() AS hoje"
).first()["hoje"]

print(f"Data inicial:        {data_min}")
print(f"Última cotação BCB:  {limites['ultima_cotacao']}")
print(f"Final do calendário: {data_max}")


# antes da 
'''
limites = (
    df_cotacao
    .agg(
        F.min("data_cotacao").alias("data_min"),
        F.max("data_cotacao").alias("data_max")
    )
    .first()
)

data_min = limites["data_min"]
data_max = limites["data_max"]

print(f"Data mínima: {data_min}")
print(f"Data máxima: {data_max}")
'''

Data inicial:        2026-09-14
Última cotação BCB:  2026-09-18
Final do calendário: 2026-09-20


'\nlimites = (\n    df_cotacao\n    .agg(\n        F.min("data_cotacao").alias("data_min"),\n        F.max("data_cotacao").alias("data_max")\n    )\n    .first()\n)\n\ndata_min = limites["data_min"]\ndata_max = limites["data_max"]\n\nprint(f"Data mínima: {data_min}")\nprint(f"Data máxima: {data_max}")\n'

In [0]:
df_calendario = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(data_min),
                F.lit(data_max),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("data_cotacao")
    )
)

df_cotacao_continua = (
    df_calendario
    .join(
        df_cotacao,
        on="data_cotacao",
        how="left"
    )
)

In [0]:
window_cotacao = (
    Window
    .orderBy("data_cotacao")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

df_cotacao_continua = (
    df_cotacao_continua
    .withColumn(
        "cotacao_compra",
        F.last(
            "cotacao_compra",
            ignorenulls=True
        ).over(window_cotacao)
    )
)

display(
    df_cotacao_continua
    .orderBy("data_cotacao")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


data_cotacao,cotacao_compra
2026-09-14,5.1690
2026-09-15,5.1484
2026-09-16,5.1520
2026-09-17,5.1515
2026-09-18,5.1569
2026-09-19,5.1569
2026-09-20,5.1569


In [0]:
total_dias = df_cotacao_continua.count()

dias_sem_cotacao = (
    df_cotacao_continua
    .filter(F.col("cotacao_compra").isNull())
    .count()
)

limites_finais = (
    df_cotacao_continua
    .agg(
        F.min("data_cotacao").alias("inicio"),
        F.max("data_cotacao").alias("fim")
    )
    .first()
)

print("VALIDAÇÃO — SÉRIE TEMPORAL DA COTAÇÃO")

print(f"Data inicial:          {limites_finais['inicio']}")
print(f"Data final:            {limites_finais['fim']}")
print(f"Quantidade de dias:    {total_dias}")
print(f"Dias sem cotação:      {dias_sem_cotacao}")

if dias_sem_cotacao == 0:
    print("[OK] Série temporal contínua e preenchida.")
else:
    print(
        "[ATENÇÃO] Ainda existem dias sem cotação."
    )

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


VALIDAÇÃO — SÉRIE TEMPORAL DA COTAÇÃO
Data inicial:          2026-09-14
Data final:            2026-09-20
Quantidade de dias:    7
Dias sem cotação:      0
[OK] Série temporal contínua e preenchida.


In [0]:
COTACAO_TARGET = (
    f"{CATALOG}.{SILVER_SCHEMA}.tb_cotacao_dolar"
)

df_cotacao_silver = (
    df_cotacao_continua
    .select(
        F.col("data_cotacao"),
        F.col("cotacao_compra")
            .cast("decimal(18,4)")
            .alias("cotacao_compra")
    )
)

(
    df_cotacao_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(COTACAO_TARGET)
)

print(f"[OK] Tabela criada/atualizada: {COTACAO_TARGET}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[OK] Tabela criada/atualizada: workspace.silver.tb_cotacao_dolar


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
df_cotacao_validacao = spark.table(COTACAO_TARGET)

print("=" * 60)
print("VALIDAÇÃO — SILVER.TB_COTACAO_DOLAR")
print("=" * 60)

print(f"Registros: {df_cotacao_validacao.count()}")

df_cotacao_validacao.printSchema()

display(
    df_cotacao_validacao
    .orderBy("data_cotacao")
)

VALIDAÇÃO — SILVER.TB_COTACAO_DOLAR
Registros: 7
root
 |-- data_cotacao: date (nullable = true)
 |-- cotacao_compra: decimal(18,4) (nullable = true)



data_cotacao,cotacao_compra
2026-09-14,5.1690
2026-09-15,5.1484
2026-09-16,5.1520
2026-09-17,5.1515
2026-09-18,5.1569
2026-09-19,5.1569
2026-09-20,5.1569


In [0]:
ultima_cotacao = (
    df_cotacao_silver
    .orderBy(F.col("data_cotacao").desc())
    .select(
        "data_cotacao",
        "cotacao_compra"
    )
    .first()
)

data_cotacao_utilizada = ultima_cotacao["data_cotacao"]
cotacao_utilizada = ultima_cotacao["cotacao_compra"]

print("=" * 60)
print("COTAÇÃO UTILIZADA NA CONVERSÃO")
print("=" * 60)

print(f"Data de referência: {data_cotacao_utilizada}")
print(f"Cotação USD/BRL:    {cotacao_utilizada}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


COTAÇÃO UTILIZADA NA CONVERSÃO
Data de referência: 2026-09-20
Cotação USD/BRL:    5.1569


In [0]:
df_financial = (
    df_financial
    .withColumn(
        "orcamento_brl",
        (
            F.col("orcamento_usd")
            * F.lit(cotacao_utilizada)
        ).cast("decimal(18,2)")
    )
    .withColumn(
        "receita_brl",
        (
            F.col("receita_usd")
            * F.lit(cotacao_utilizada)
        ).cast("decimal(18,2)")
    )
)

In [0]:
df_financial = (
    df_financial
    .withColumn(
        "lucro_usd",
        (
            F.col("receita_usd")
            - F.col("orcamento_usd")
        ).cast("decimal(18,2)")
    )
    .withColumn(
        "lucro_brl",
        (
            F.col("receita_brl")
            - F.col("orcamento_brl")
        ).cast("decimal(18,2)")
    )
)

In [0]:
df_financial = (
    df_financial
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            F.col("receita_usd").isNotNull()
            & F.col("orcamento_usd").isNotNull()
            & (F.col("receita_usd") > 0),

            F.round(
                (
                    F.col("lucro_usd")
                    / F.col("receita_usd")
                ) * 100,
                2
            )
        )
        .otherwise(F.lit(None))
        .cast("decimal(10,2)")
    )
)

In [0]:
df_financial_final = df_financial.select(
    "id_filme",
    F.col("orcamento_usd").cast("decimal(18,2)"),
    F.col("receita_usd").cast("decimal(18,2)"),
    F.col("orcamento_brl").cast("decimal(18,2)"),
    F.col("receita_brl").cast("decimal(18,2)"),
    F.col("lucro_usd").cast("decimal(18,2)"),
    F.col("lucro_brl").cast("decimal(18,2)"),
    F.col("margem_lucro_percentual")
)

df_financial_final.printSchema()

display(
    df_financial_final
    .filter(
        F.col("receita_usd").isNotNull()
        & F.col("orcamento_usd").isNotNull()
    )
    .limit(20)
)

root
 |-- id_filme: string (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- margem_lucro_percentual: decimal(10,2) (nullable = true)



id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
1001724,3.00,8000.00,15.47,41255.20,7997.00,41239.73,99.96
1003578,10000.00,3000000.00,51569.00,15470700.00,2990000.00,15419131.00,99.67
1008042,4500000.00,72600000.00,23206050.00,374390940.00,68100000.00,351184890.00,93.80
1009615,4.00,10000.00,20.63,51569.00,9996.00,51548.37,99.96
1012642,25.00,123.00,128.92,634.30,98.00,505.38,79.67
1014127,500.00,10000.00,2578.45,51569.00,9500.00,48990.55,95.00
1014193,10.00,200.00,51.57,1031.38,190.00,979.81,95.00
1024777,10000.00,60000.00,51569.00,309414.00,50000.00,257845.00,83.33
1026837,40000.00,800.00,206276.00,4125.52,-39200.00,-202150.48,-4900.00
1028194,1.00,400.00,5.16,2062.76,399.00,2057.60,99.75


In [0]:
total = df_financial_final.count()

ids_distintos = (
    df_financial_final
    .select("id_filme")
    .distinct()
    .count()
)

orcamento_invalido = (
    df_financial_final
    .filter(F.col("orcamento_usd") <= 0)
    .count()
)

receita_invalida = (
    df_financial_final
    .filter(F.col("receita_usd") <= 0)
    .count()
)

print("=" * 60)
print("VALIDAÇÃO — TB_FINANCEIRO_FILMES")
print("=" * 60)

print(f"Total de registros:        {total}")
print(f"IDs distintos:             {ids_distintos}")
print(f"Orçamentos <= 0:           {orcamento_invalido}")
print(f"Receitas <= 0:             {receita_invalida}")

if total == ids_distintos:
    print("[OK] Não existem IDs duplicados.")
else:
    print("[ERRO] Existem IDs duplicados.")

if orcamento_invalido == 0 and receita_invalida == 0:
    print("[OK] Não existem valores financeiros inválidos.")

VALIDAÇÃO — TB_FINANCEIRO_FILMES
Total de registros:        99006
IDs distintos:             99006
Orçamentos <= 0:           0
Receitas <= 0:             0
[OK] Não existem IDs duplicados.
[OK] Não existem valores financeiros inválidos.


In [0]:
(
    df_financial_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FINANCIAL_TARGET)
)

print(
    f"[OK] Tabela criada/atualizada: "
    f"{FINANCIAL_TARGET}"
)

[OK] Tabela criada/atualizada: workspace.silver.tb_financeiro_filmes


In [0]:
df_financial_validacao = spark.table(FINANCIAL_TARGET)

print("=" * 60)
print("VALIDAÇÃO FINAL — SILVER.TB_FINANCEIRO_FILMES")
print("=" * 60)

print(f"Tabela: {FINANCIAL_TARGET}")
print(f"Registros: {df_financial_validacao.count()}")

df_financial_validacao.printSchema()

display(
    df_financial_validacao
    .filter(
        F.col("receita_usd").isNotNull()
        & F.col("orcamento_usd").isNotNull()
    )
    .limit(20)
)

VALIDAÇÃO FINAL — SILVER.TB_FINANCEIRO_FILMES
Tabela: workspace.silver.tb_financeiro_filmes
Registros: 99006
root
 |-- id_filme: string (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- margem_lucro_percentual: decimal(10,2) (nullable = true)



id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
1001724,3.00,8000.00,15.47,41255.20,7997.00,41239.73,99.96
1003578,10000.00,3000000.00,51569.00,15470700.00,2990000.00,15419131.00,99.67
1008042,4500000.00,72600000.00,23206050.00,374390940.00,68100000.00,351184890.00,93.80
1009615,4.00,10000.00,20.63,51569.00,9996.00,51548.37,99.96
1012642,25.00,123.00,128.92,634.30,98.00,505.38,79.67
1014127,500.00,10000.00,2578.45,51569.00,9500.00,48990.55,95.00
1014193,10.00,200.00,51.57,1031.38,190.00,979.81,95.00
1024777,10000.00,60000.00,51569.00,309414.00,50000.00,257845.00,83.33
1026837,40000.00,800.00,206276.00,4125.52,-39200.00,-202150.48,-4900.00
1028194,1.00,400.00,5.16,2062.76,399.00,2057.60,99.75


In [0]:
METRICS_SOURCE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.tb_movies_metrics"
)

METRICS_TARGET = (
    f"{CATALOG}.{SILVER_SCHEMA}.tb_metricas_engajamento"
)

df_metrics_raw = spark.table(METRICS_SOURCE)

print(f"Origem: {METRICS_SOURCE}")
print(f"Registros: {df_metrics_raw.count()}")

df_metrics_raw.printSchema()

display(
    df_metrics_raw.limit(20)
)

Origem: workspace.bronze.tb_movies_metrics
Registros: 107364
root
 |-- id: string (nullable = true)
 |-- popularity: string (nullable = true)
 |-- vote_average: string (nullable = true)
 |-- vote_count: string (nullable = true)
 |-- averageRating: string (nullable = true)
 |-- numVotes: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



id,popularity,vote_average,vote_count,averageRating,numVotes,ingestion_datetime
293660,72.735,7.606,28894,8.0,1270339,2026-09-19T17:52:10.027Z
299536,"154,34",8.255,27713,8.4,1406782,2026-09-19T17:52:10.027Z
299534,91.756,8.263,23857,8.4,1484150,2026-09-19T17:52:10.027Z
475557,"54,522",8.168,23425,8.3,1723035,2026-09-19T17:52:10.027Z
271110,70.741,7.4,21541,7.8,947222,2026-09-19T17:52:10.027Z
284054,43.665,7.39,null,7.3,924922,2026-09-19T17:52:10.027Z
284052,70.535,7.427,20935,7.5,895880,2026-09-19T17:52:10.027Z
315635,65.88,7.345,20507,7.4,835116,2026-09-19T17:52:10.027Z
283995,67.553,7.624,20353,7.6,844767,2026-09-19T17:52:10.027Z
297761,35.356,5.909,20097,5.9,null,2026-09-19T17:52:10.027Z


In [0]:
display(
    df_metrics_raw
    .select("id", "popularity")
    .where(F.col("popularity").isNotNull())
    .distinct()
    .limit(100)
)

id,popularity
293660,72.735
299536,"154,34"
299534,91.756
475557,"54,522"
271110,70.741
284054,43.665
284052,70.535
315635,65.88
283995,67.553
297761,35.356


In [0]:
display(
    df_metrics_raw
    .select(
        "id",
        "vote_average",
        "vote_count"
    )
    .where(
        F.col("vote_average").isNotNull()
        | F.col("vote_count").isNotNull()
    )
    .limit(100)
)

id,vote_average,vote_count
293660,7.606,28894
299536,8.255,27713
299534,8.263,23857
475557,8.168,23425
271110,7.4,21541
284054,7.39,null
284052,7.427,20935
315635,7.345,20507
283995,7.624,20353
297761,5.909,20097


In [0]:
display(
    df_metrics_raw
    .select(
        "id",
        "averageRating",
        "numVotes"
    )
    .where(
        F.col("averageRating").isNotNull()
        | F.col("numVotes").isNotNull()
    )
    .limit(100)
)

id,averageRating,numVotes
293660,8.0,1270339
299536,8.4,1406782
299534,8.4,1484150
475557,8.3,1723035
271110,7.8,947222
284054,7.3,924922
284052,7.5,895880
315635,7.4,835116
283995,7.6,844767
297761,5.9,null


In [0]:
display(
    df_metrics_raw
    .filter(
        F.col("vote_average").rlike("[A-Za-z]")
        |
        F.col("vote_count").rlike("[A-Za-z]")
        |
        F.col("averageRating").rlike("[A-Za-z]")
        |
        F.col("numVotes").rlike("[A-Za-z]")
    )
    .select(
        "id",
        "popularity",
        "vote_average",
        "vote_count",
        "averageRating",
        "numVotes"
    )
    .limit(100)
)

id,popularity,vote_average,vote_count,averageRating,numVotes
324857,"causing others from across the Spider-Verse to be inadvertently transported to his dimension.""",8.404,null,"Phil Lord""","Rodney Rothman"""""
301528,"a road trip adventure alongside old and new friends will show Woody how big the world can be for a toy.""",7.5,9093,"John Lasseter""",Andrew Stanton
615457,"Hutch's unknown long-simmering rage is ignited and propels him on a brutal path that will uncover dark secrets he fought to leave behind,""",8.018,6217,Derek Kolstad,7.4
140300,Po must face two hugely epic,6.878,5327,"Alessandro Carloni, Jennifer Yuh Nelson","Jonathan Aibel, Glenn Berger, Ethan Reiff, Cyrus Voris"
397243,"""\"""" they discover increasingly bizarre clues that hold the key to her terrifying secrets.\""""""""""",6.755,null,"Ian Goldberg, Richard Naing",6.8
364689,fastest,7.232,2652,Carlos Saldanha,"Robert L. Baird, Tim Federle, Brad Copeland, Ron Burch, David Kidd, Don Rhymer, Munro Leaf, Robert Lawson"
594767,are forced to get back into action and fight the Daughters of Atlas,6.662,2416,David F. Sandberg,"Henry Gayden, Chris Morgan, Bill Parker, C.C. Beck, William Moulton Marston"
505948,"designed to repopulate the earth following an extinction event. But their unique bond is threatened when an inexplicable stranger arrives with alarming news.""",6.684,2345,"Michael Lloyd Green""","Grant Sputore"""""
602269,Baxter is unaware that the investigation is dredging up echoes of Deke's past,6.329,2158,John Lee Hancock,John Lee Hancock
585083,"""\"""" goes haywire""""""",6.998,null,United States of America,English


In [0]:
display(
    df_metrics_raw
    .select(
        "id",
        "vote_average",
        "vote_count",
        "averageRating",
        "numVotes"
    )
    .filter(
        (
            F.expr(
                "try_cast(vote_average AS DOUBLE)"
            ) > 10
        )
        |
        (
            F.expr(
                "try_cast(vote_average AS DOUBLE)"
            ) < 0
        )
        |
        (
            F.expr(
                "try_cast(averageRating AS DOUBLE)"
            ) > 10
        )
        |
        (
            F.expr(
                "try_cast(averageRating AS DOUBLE)"
            ) < 0
        )
        |
        (
            F.expr(
                "try_cast(vote_count AS BIGINT)"
            ) < 0
        )
        |
        (
            F.expr(
                "try_cast(numVotes AS BIGINT)"
            ) < 0
        )
    )
    .limit(100)
)

id,vote_average,vote_count,averageRating,numVotes
424694,79.97,15965,null,653883
321612,69.67,14871,7.1,362493
141052,60.98,12263,6.0,509478
530915,79.9,11395,8.2,784874
490132,82.42000000000002,null,8.2,702296
76600,76.53999999999999,9830,7.5,646896
181812,63.650000000000006,9047,6.3,550342
337404,80.55,8507,7.3,301069
337167,67.05,7511,4.6,84532
241259,65.5,6025,6.2,136921


In [0]:
df_metrics_raw.select(
    F.count("*").alias("total"),

    F.sum(
        F.when(
            F.expr(
                "try_cast(vote_average AS DOUBLE) IS NULL"
            )
            & F.col("vote_average").isNotNull(),
            1
        ).otherwise(0)
    ).alias("vote_average_nao_numerico"),

    F.sum(
        F.when(
            F.expr(
                "try_cast(vote_count AS BIGINT) IS NULL"
            )
            & F.col("vote_count").isNotNull(),
            1
        ).otherwise(0)
    ).alias("vote_count_nao_numerico"),

    F.sum(
        F.when(
            F.expr(
                "try_cast(averageRating AS DOUBLE) IS NULL"
            )
            & F.col("averageRating").isNotNull(),
            1
        ).otherwise(0)
    ).alias("averageRating_nao_numerico"),

    F.sum(
        F.when(
            F.expr(
                "try_cast(numVotes AS BIGINT) IS NULL"
            )
            & F.col("numVotes").isNotNull(),
            1
        ).otherwise(0)
    ).alias("numVotes_nao_numerico")

).show(truncate=False)

+------+-------------------------+-----------------------+--------------------------+---------------------+
|total |vote_average_nao_numerico|vote_count_nao_numerico|averageRating_nao_numerico|numVotes_nao_numerico|
+------+-------------------------+-----------------------+--------------------------+---------------------+
|107364|582                      |579                    |2684                      |3528                 |
+------+-------------------------+-----------------------+--------------------------+---------------------+



In [0]:
df_metrics = df_metrics_raw.select(
    F.col("id").cast("string").alias("id_filme"),
    F.col("popularity").alias("popularidade_raw"),
    F.col("vote_average").alias("nota_media_tmdb_raw"),
    F.col("vote_count").alias("qtd_votos_tmdb_raw"),
    F.col("averageRating").alias("nota_media_imdb_raw"),
    F.col("numVotes").alias("qtd_votos_imdb_raw"),
    F.col("ingestion_datetime")
)

In [0]:
def limpar_popularidade(coluna):
    valor = F.trim(F.col(coluna))

    valor_normalizado = F.regexp_replace(
        valor,
        ",",
        "."
    )

    valor_numerico = F.expr(
        f"""
        try_cast(
            regexp_replace(trim({coluna}), ',', '.')
            AS DOUBLE
        )
        """
    )

    return (
        F.when(
            valor_numerico >= 0,
            valor_numerico
        )
        .otherwise(F.lit(None))
        .cast("double")
    )

In [0]:
def limpar_nota(coluna):
    valor_numerico = F.expr(
        f"""
        try_cast(
            regexp_replace(trim({coluna}), ',', '.')
            AS DOUBLE
        )
        """
    )

    return (
        F.when(
            valor_numerico.between(0, 10),
            valor_numerico
        )
        .otherwise(F.lit(None))
        .cast("double")
    )

In [0]:
def limpar_quantidade_votos(coluna):
    valor = F.trim(F.col(coluna))

    valor_numerico = F.expr(
        f"""
        try_cast(
            trim({coluna})
            AS BIGINT
        )
        """
    )

    return (
        F.when(
            valor_numerico >= 0,
            valor_numerico
        )
        .otherwise(F.lit(None))
        .cast("long")
    )

In [0]:
df_metrics = (
    df_metrics
    .withColumn(
        "popularidade",
        limpar_popularidade("popularidade_raw")
    )
    .withColumn(
        "nota_media_tmdb",
        limpar_nota("nota_media_tmdb_raw")
    )
    .withColumn(
        "qtd_votos_tmdb",
        limpar_quantidade_votos("qtd_votos_tmdb_raw")
    )
    .withColumn(
        "nota_media_imdb",
        limpar_nota("nota_media_imdb_raw")
    )
    .withColumn(
        "qtd_votos_imdb",
        limpar_quantidade_votos("qtd_votos_imdb_raw")
    )
)

In [0]:
display(
    df_metrics
    .filter(
        F.col("popularidade_raw").rlike("[A-Za-z]")
        |
        F.col("nota_media_tmdb_raw").rlike("[A-Za-z]")
        |
        F.col("qtd_votos_tmdb_raw").rlike("[A-Za-z]")
        |
        F.col("nota_media_imdb_raw").rlike("[A-Za-z]")
        |
        F.col("qtd_votos_imdb_raw").rlike("[A-Za-z]")
    )
    .select(
        "id_filme",
        "popularidade_raw",
        "popularidade",
        "nota_media_tmdb_raw",
        "nota_media_tmdb",
        "qtd_votos_tmdb_raw",
        "qtd_votos_tmdb",
        "nota_media_imdb_raw",
        "nota_media_imdb",
        "qtd_votos_imdb_raw",
        "qtd_votos_imdb"
    )
    .limit(100)
)

id_filme,popularidade_raw,popularidade,nota_media_tmdb_raw,nota_media_tmdb,qtd_votos_tmdb_raw,qtd_votos_tmdb,nota_media_imdb_raw,nota_media_imdb,qtd_votos_imdb_raw,qtd_votos_imdb
324857,"causing others from across the Spider-Verse to be inadvertently transported to his dimension.""",null,8.404,8.404,null,null,"Phil Lord""",null,"Rodney Rothman""""",null
301528,"a road trip adventure alongside old and new friends will show Woody how big the world can be for a toy.""",null,7.5,7.5,9093,9093,"John Lasseter""",null,Andrew Stanton,null
615457,"Hutch's unknown long-simmering rage is ignited and propels him on a brutal path that will uncover dark secrets he fought to leave behind,""",null,8.018,8.018,6217,6217,Derek Kolstad,null,7.4,null
140300,Po must face two hugely epic,null,6.878,6.878,5327,5327,"Alessandro Carloni, Jennifer Yuh Nelson",null,"Jonathan Aibel, Glenn Berger, Ethan Reiff, Cyrus Voris",null
397243,"""\"""" they discover increasingly bizarre clues that hold the key to her terrifying secrets.\""""""""""",null,6.755,6.755,null,null,"Ian Goldberg, Richard Naing",null,6.8,null
364689,fastest,null,7.232,7.232,2652,2652,Carlos Saldanha,null,"Robert L. Baird, Tim Federle, Brad Copeland, Ron Burch, David Kidd, Don Rhymer, Munro Leaf, Robert Lawson",null
594767,are forced to get back into action and fight the Daughters of Atlas,null,6.662,6.662,2416,2416,David F. Sandberg,null,"Henry Gayden, Chris Morgan, Bill Parker, C.C. Beck, William Moulton Marston",null
505948,"designed to repopulate the earth following an extinction event. But their unique bond is threatened when an inexplicable stranger arrives with alarming news.""",null,6.684,6.684,2345,2345,"Michael Lloyd Green""",null,"Grant Sputore""""",null
602269,Baxter is unaware that the investigation is dredging up echoes of Deke's past,null,6.329,6.329,2158,2158,John Lee Hancock,null,John Lee Hancock,null
585083,"""\"""" goes haywire""""""",null,6.998,6.998,null,null,United States of America,null,English,null


In [0]:
metricas_invalidas = df_metrics.filter(
    (F.col("popularidade") < 0)
    |
    (~F.col("nota_media_tmdb").between(0, 10))
    |
    (F.col("qtd_votos_tmdb") < 0)
    |
    (~F.col("nota_media_imdb").between(0, 10))
    |
    (F.col("qtd_votos_imdb") < 0)
).count()

print("=" * 60)
print("VALIDAÇÃO — REGRAS DAS MÉTRICAS")
print("=" * 60)

print(
    f"Métricas inválidas restantes: "
    f"{metricas_invalidas}"
)

if metricas_invalidas == 0:
    print(
        "[OK] Todas as métricas respeitam "
        "os limites definidos."
    )
else:
    print(
        "[ERRO] Ainda existem métricas inválidas."
    )

VALIDAÇÃO — REGRAS DAS MÉTRICAS
Métricas inválidas restantes: 0
[OK] Todas as métricas respeitam os limites definidos.


In [0]:
window_metrics = (
    Window
    .partitionBy("id_filme")
    .orderBy(
        F.col("ingestion_datetime").desc()
    )
)

df_metrics = (
    df_metrics
    .withColumn(
        "_row_number",
        F.row_number().over(window_metrics)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

In [0]:
df_metrics_final = df_metrics.select(
    "id_filme",
    F.col("popularidade")
        .cast("double")
        .alias("popularidade"),
    F.col("nota_media_tmdb")
        .cast("double")
        .alias("nota_media_tmdb"),
    F.col("qtd_votos_tmdb")
        .cast("int")
        .alias("qtd_votos_tmdb"),
    F.col("nota_media_imdb")
        .cast("double")
        .alias("nota_media_imdb"),
    F.col("qtd_votos_imdb")
        .cast("int")
        .alias("qtd_votos_imdb")
)

In [0]:
total_metrics = df_metrics_final.count()

ids_metrics = (
    df_metrics_final
    .select("id_filme")
    .distinct()
    .count()
)

ids_nulos = (
    df_metrics_final
    .filter(F.col("id_filme").isNull())
    .count()
)

print("=" * 60)
print("VALIDAÇÃO — TB_METRICAS_ENGAJAMENTO")
print("=" * 60)

print(f"Total de registros: {total_metrics}")
print(f"IDs distintos:      {ids_metrics}")
print(f"IDs nulos:           {ids_nulos}")

if total_metrics == ids_metrics:
    print("[OK] Não existem IDs duplicados.")
else:
    print("[ERRO] Existem IDs duplicados.")

VALIDAÇÃO — TB_METRICAS_ENGAJAMENTO
Total de registros: 99013
IDs distintos:      99013
IDs nulos:           0
[OK] Não existem IDs duplicados.


In [0]:
(
    df_metrics_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(METRICS_TARGET)
)

print(
    f"[OK] Tabela criada/atualizada: "
    f"{METRICS_TARGET}"
)

[OK] Tabela criada/atualizada: workspace.silver.tb_metricas_engajamento


In [0]:
df_metrics_validacao = spark.table(
    METRICS_TARGET
)

print("=" * 60)
print(
    "VALIDAÇÃO FINAL — "
    "SILVER.TB_METRICAS_ENGAJAMENTO"
)
print("=" * 60)

print(
    f"Tabela: {METRICS_TARGET}"
)

print(
    f"Registros: "
    f"{df_metrics_validacao.count()}"
)

df_metrics_validacao.printSchema()

display(
    df_metrics_validacao.limit(20)
)

VALIDAÇÃO FINAL — SILVER.TB_METRICAS_ENGAJAMENTO
Tabela: workspace.silver.tb_metricas_engajamento
Registros: 99013
root
 |-- id_filme: string (nullable = true)
 |-- popularidade: double (nullable = true)
 |-- nota_media_tmdb: double (nullable = true)
 |-- qtd_votos_tmdb: integer (nullable = true)
 |-- nota_media_imdb: double (nullable = true)
 |-- qtd_votos_imdb: integer (nullable = true)



id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1000004,1.132,0.0,0,6.8,27
1000005,0.6,0.0,0,null,40
1000007,0.6,0.0,0,4.7,10
1000011,1.169,0.0,0,4.8,16
1000014,0.6,0.0,0,7.2,15
1000030,0.615,0.0,0,null,25
1000054,0.6,0.0,0,5.9,10
1000058,1.489,6.75,6,6.2,382
1000059,0.6,0.0,0,7.7,25
1000073,13.212,6.8,15,null,236


In [0]:
REVIEWS_SOURCE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.tb_movies_reviews"
)

REVIEWS_TARGET = (
    f"{CATALOG}.{SILVER_SCHEMA}.tb_avaliacoes_usuarios"
)

df_reviews_raw = spark.table(REVIEWS_SOURCE)

print(f"Origem: {REVIEWS_SOURCE}")
print(f"Registros: {df_reviews_raw.count()}")

df_reviews_raw.printSchema()

display(df_reviews_raw.limit(20))

Origem: workspace.bronze.tb_movies_reviews
Registros: 32412
root
 |-- id: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- nota: string (nullable = true)
 |-- comentario: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



id,nome,nota,comentario,ingestion_datetime
442113,Mariana Cardoso 277,4.4,null,2026-09-19T17:52:20.072Z
637007,Lucas Reis 602,3.9,null,2026-09-19T17:52:20.072Z
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.,2026-09-19T17:52:20.072Z
413036,Gabriela Monteiro 401,7.5,null,2026-09-19T17:52:20.072Z
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.,2026-09-19T17:52:20.072Z
387727,Maria Alves 707,8.2,"Gostei bastante, recomendo.",2026-09-19T17:52:20.072Z
1032506,Amanda Castro 242,4.7,"Fraco, não recomendo.",2026-09-19T17:52:20.072Z
446554,Sandra Rodrigues 736,6.5,Assisti até o final mas não me marcou.,2026-09-19T17:52:20.072Z
569916,Camila Lima 489,8.2,Muito bom! Vale a pena assistir.,2026-09-19T17:52:20.072Z
864552,Roberto Almeida 555,9.3,"Obra-prima do cinema, simplesmente espetacular.",2026-09-19T17:52:20.072Z


In [0]:
display(
    df_reviews_raw
    .select(
        "id",
        "nome",
        "nota",
        "comentario"
    )
    .limit(100)
)

id,nome,nota,comentario
442113,Mariana Cardoso 277,4.4,null
637007,Lucas Reis 602,3.9,null
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.
413036,Gabriela Monteiro 401,7.5,null
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.
387727,Maria Alves 707,8.2,"Gostei bastante, recomendo."
1032506,Amanda Castro 242,4.7,"Fraco, não recomendo."
446554,Sandra Rodrigues 736,6.5,Assisti até o final mas não me marcou.
569916,Camila Lima 489,8.2,Muito bom! Vale a pena assistir.
864552,Roberto Almeida 555,9.3,"Obra-prima do cinema, simplesmente espetacular."


In [0]:
display(
    df_reviews_raw
    .select("nota")
    .where(F.col("nota").isNotNull())
    .distinct()
    .orderBy("nota")
)

nota
0.0
0.1
0.2
0.3
0.4
0.5
0.6
0.7
0.8
0.9


In [0]:
display(
    df_reviews_raw
    .filter(
        (
            F.expr(
                "try_cast(regexp_replace(trim(nota), ',', '.') AS DOUBLE)"
            ).isNull()
            & F.col("nota").isNotNull()
        )
        |
        (
            F.expr(
                "try_cast(regexp_replace(trim(nota), ',', '.') AS DOUBLE)"
            ) < 0
        )
        |
        (
            F.expr(
                "try_cast(regexp_replace(trim(nota), ',', '.') AS DOUBLE)"
            ) > 10
        )
    )
    .select(
        "id",
        "nome",
        "nota",
        "comentario"
    )
    .limit(100)
)

id,nome,nota,comentario


In [0]:
comentarios_nulos = (
    df_reviews_raw
    .filter(F.col("comentario").isNull())
    .count()
)

comentarios_vazios = (
    df_reviews_raw
    .filter(
        F.col("comentario").isNotNull()
        & (F.trim(F.col("comentario")) == "")
    )
    .count()
)

print("=" * 60)
print("ANÁLISE — COMENTÁRIOS")
print("=" * 60)

print(f"Comentários NULL:   {comentarios_nulos}")
print(f"Comentários vazios: {comentarios_vazios}")

ANÁLISE — COMENTÁRIOS
Comentários NULL:   7776
Comentários vazios: 0


In [0]:
duplicatas_reviews = (
    df_reviews_raw
    .groupBy(
        "id",
        "nome",
        "nota",
        "comentario"
    )
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Combinações duplicadas: "
    f"{duplicatas_reviews.count()}"
)

display(
    duplicatas_reviews
    .orderBy(F.col("count").desc())
    .limit(100)
)

Combinações duplicadas: 0


id,nome,nota,comentario,count


In [0]:
df_reviews = df_reviews_raw.select(
    F.col("id").cast("string").alias("id_filme"),
    F.col("nome").alias("nome_usuario"),
    F.col("nota").alias("nota_raw"),
    F.col("comentario").alias("comentario_usuario")
)

In [0]:
nota_numerica = F.expr(
    """
    try_cast(
        regexp_replace(trim(nota_raw), ',', '.')
        AS DOUBLE
    )
    """
)

df_reviews = (
    df_reviews
    .withColumn(
        "nota_usuario",
        F.when(
            nota_numerica.between(0, 10),
            nota_numerica
        )
        .otherwise(F.lit(None))
        .cast("double")
    )
)

In [0]:
df_reviews = (
    df_reviews
    .withColumn(
        "comentario_usuario",
        F.when(
            F.col("comentario_usuario").isNull()
            |
            (F.trim(F.col("comentario_usuario")) == ""),
            F.lit("Sem comentário")
        )
        .otherwise(F.trim(F.col("comentario_usuario")))
    )
)

In [0]:
df_reviews = (
    df_reviews
    .dropDuplicates([
        "id_filme",
        "nome_usuario",
        "nota_usuario",
        "comentario_usuario"
    ])
)

In [0]:
df_reviews_final = df_reviews.select(
    "id_filme",
    "nome_usuario",
    "nota_usuario",
    "comentario_usuario"
)

In [0]:
total_reviews = df_reviews_final.count()

notas_invalidas = (
    df_reviews_final
    .filter(
        (F.col("nota_usuario") < 0)
        |
        (F.col("nota_usuario") > 10)
    )
    .count()
)

comentarios_invalidos = (
    df_reviews_final
    .filter(
        F.col("comentario_usuario").isNull()
        |
        (F.trim(F.col("comentario_usuario")) == "")
    )
    .count()
)

duplicatas_finais = (
    df_reviews_final
    .groupBy(
        "id_filme",
        "nome_usuario",
        "nota_usuario",
        "comentario_usuario"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

sem_comentario = (
    df_reviews_final
    .filter(
        F.col("comentario_usuario") == "Sem comentário"
    )
    .count()
)

print("=" * 60)
print("VALIDAÇÃO — TB_AVALIACOES_USUARIOS")
print("=" * 60)

print(f"Total de registros:          {total_reviews}")
print(f"Notas inválidas restantes:   {notas_invalidas}")
print(f"Comentários inválidos:       {comentarios_invalidos}")
print(f"Duplicatas restantes:        {duplicatas_finais}")
print(f'Registros "Sem comentário":  {sem_comentario}')

if (
    notas_invalidas == 0
    and comentarios_invalidos == 0
    and duplicatas_finais == 0
):
    print("[OK] Todas as regras de avaliações foram atendidas.")
else:
    print("[ERRO] Existem inconsistências nas avaliações.")

VALIDAÇÃO — TB_AVALIACOES_USUARIOS
Total de registros:          32412
Notas inválidas restantes:   0
Comentários inválidos:       0
Duplicatas restantes:        0
Registros "Sem comentário":  7776
[OK] Todas as regras de avaliações foram atendidas.


In [0]:
(
    df_reviews_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(REVIEWS_TARGET)
)

print(
    f"[OK] Tabela criada/atualizada: "
    f"{REVIEWS_TARGET}"
)

[OK] Tabela criada/atualizada: workspace.silver.tb_avaliacoes_usuarios


In [0]:
df_reviews_validacao = spark.table(
    REVIEWS_TARGET
)

print("=" * 60)
print(
    "VALIDAÇÃO FINAL — "
    "SILVER.TB_AVALIACOES_USUARIOS"
)
print("=" * 60)

print(f"Tabela: {REVIEWS_TARGET}")
print(f"Registros: {df_reviews_validacao.count()}")

df_reviews_validacao.printSchema()

display(
    df_reviews_validacao.limit(20)
)

VALIDAÇÃO FINAL — SILVER.TB_AVALIACOES_USUARIOS
Tabela: workspace.silver.tb_avaliacoes_usuarios
Registros: 32412
root
 |-- id_filme: string (nullable = true)
 |-- nome_usuario: string (nullable = true)
 |-- nota_usuario: double (nullable = true)
 |-- comentario_usuario: string (nullable = true)



id_filme,nome_usuario,nota_usuario,comentario_usuario
637007,Lucas Reis 602,3.9,Sem comentário
1100094,Gabriel Carvalho 581,6.2,"Aceitável, mas esperava mais."
628575,Alexandre Barbosa 220,0.3,Péssimo em todos os sentidos.
573249,Rodrigo Oliveira 273,0.5,Péssimo em todos os sentidos.
592539,Pedro Costa 181,4.8,"Não gostei, história confusa."
464493,Adriana Dias 257,0.4,Péssimo em todos os sentidos.
1199748,Eduardo Dias 177,6.0,"Poderia ser melhor, mas não é ruim."
640543,Cristina Monteiro 310,6.4,Sem comentário
599134,Larissa Lopes 330,2.6,Péssimo em todos os sentidos.
424011,Vinícius Ferreira 405,0.5,Não recomendo de jeito nenhum.


In [0]:
CREDITS_SOURCE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.tb_credits_and_tags"
)

df_credits_raw = spark.table(CREDITS_SOURCE)

print(f"Origem: {CREDITS_SOURCE}")
print(f"Registros: {df_credits_raw.count()}")

df_credits_raw.printSchema()

display(df_credits_raw.limit(20))

Origem: workspace.bronze.tb_credits_and_tags
Registros: 106320
root
 |-- id: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_companies: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- spoken_languages: string (nullable = true)
 |-- keywords: string (nullable = true)
 |-- directors: string (nullable = true)
 |-- writers: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



id,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,cast,ingestion_datetime
293660,"Action, Adventure, Comedy","20th Century Fox, The Donners' Company, Genre Films",United States of America,English,"superhero, anti hero, mercenary, based on comic, aftercreditsstinger, duringcreditsstinger",Tim Miller,"Rhett Reese, Paul Wernick","Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder",2026-09-19T17:52:15.268Z
299536,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, space, battlefield, genocide, magical object, super power, aftercreditsstinger, marvel cinematic universe (mcu), cosmic","Anthony Russo, Joe Russo",N/A,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman",2026-09-19T17:52:15.268Z
299534,"Adventure, Science Fiction, Action",Marvel Studios,United States of America,"English, Japanese, Xhosa","superhero, time travel, space travel, time machine, based on comic, sequel, alien invasion, superhero team, marvel cinematic universe (mcu), alternate timeline, father daughter relationship, sister sister relationship","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee, Jack Kirby, Joe Simon, Steve Englehart, Steve Gan, Bill Mantlo, Keith Giffen, Jim Starlin, Larry Lieber, Don Heck","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Josh Brolin, Don Cheadle, Paul Rudd, Benedict Cumberbatch",2026-09-19T17:52:15.268Z
475557,"Crime, Thriller, Drama","Warner Bros. Pictures, Joint Effort, Village Roadshow Pictures, Bron Studios, DC Films","Canada, United States of America",English,"dream, street gang, society, psychopath, clown, villain, based on comic, murder, psychological thriller, criminal mastermind, mental illness, anarchy, character study, clown makeup, subway train, social realism, supervillain, tv host, 1980s, mother son relationship, origin story, falling into madness, depressing",Todd Phillips,"Todd Phillips, Scott Silver, Bob Kane, Bill Finger, Jerry Robinson","Joaquin Phoenix, Robert De Niro, Zazie Beetz, Frances Conroy, Brett Cullen, Shea Whigham, Bill Camp, Glenn Fleshler, Leigh Gill, Josh Pais",2026-09-19T17:52:15.268Z
271110,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"Romanian, English, German, Russian","civil war, superhero, based on comic, sequel, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu), excited","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Joe Simon, Jack Kirby","Chris Evans, Robert Downey Jr., Scarlett Johansson, Sebastian Stan, Anthony Mackie, Don Cheadle, Jeremy Renner, Chadwick Boseman, Paul Bettany, Elizabeth Olsen",2026-09-19T17:52:15.268Z
284054,"Action, Adventure, Science Fiction",Marvel Studios,United States of America,"English, Korean, Swahili, Xhosa","africa, superhero, based on comic, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu)",Ryan Coogler,"Ryan Coogler, Joe Robert Cole, Stan Lee, Jack Kirby","Chadwick Boseman, Michael B. Jordan, Lupita Nyong'o, Danai Gurira, Martin Freeman, Daniel Kaluuya, Letitia Wright, Winston Duke, Sterling K. Brown, Angela Bassett",2026-09-19T17:52:15.268Z
284052,"Action, Adventure, Fantasy",Marvel Studios,UNITED STATES OF AMERICA,English,"magic, superhero, training, time, based on comic, sorcerer, doctor, neurosurgeon, wizard, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu)",Scott Derrickson,N/A,"Benedict Cumberbatch, Chiwetel Ejiofor, Rachel McAdams, Benedict Wong, Mads Mikkelsen, Tilda Swinton, Michael Stuhlbarg, Benjamin Bratt, Scott Adkins, Zara Phythian",2026-09-19T17:52:15.268Z
315635,"Action, Adventure, Science Fiction, Drama","Ma

In [0]:
display(
    df_credits_raw
    .select(
        "id",
        "genres"
    )
    .where(F.col("genres").isNotNull())
    .limit(100)
)

id,genres
293660,"Action, Adventure, Comedy"
299536,"Adventure, Action, Science Fiction"
299534,"Adventure, Science Fiction, Action"
475557,"Crime, Thriller, Drama"
271110,"Adventure, Action, Science Fiction"
284054,"Action, Adventure, Science Fiction"
284052,"Action, Adventure, Fantasy"
315635,"Action, Adventure, Science Fiction, Drama"
283995,"Science Fiction, Adventure, Action"
297761,Action|Adventure|Fantasy


In [0]:
display(
    df_credits_raw
    .select("genres")
    .where(F.col("genres").isNotNull())
    .distinct()
    .limit(100)
)

genres
"Action, Adventure, Comedy"
"Adventure, Action, Science Fiction"
"Adventure, Science Fiction, Action"
"Crime, Thriller, Drama"
"Action, Adventure, Science Fiction"
"Action, Adventure, Fantasy"
"Action, Adventure, Science Fiction, Drama"
"Science Fiction, Adventure, Action"
Action|Adventure|Fantasy
"Action, Drama, Science Fiction"


In [0]:
display(
    df_credits_raw
    .select(
        "id",
        "cast"
    )
    .where(F.col("cast").isNotNull())
    .limit(100)
)

id,cast
293660,"Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder"
299536,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman"
299534,"Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Josh Brolin, Don Cheadle, Paul Rudd, Benedict Cumberbatch"
475557,"Joaquin Phoenix, Robert De Niro, Zazie Beetz, Frances Conroy, Brett Cullen, Shea Whigham, Bill Camp, Glenn Fleshler, Leigh Gill, Josh Pais"
271110,"Chris Evans, Robert Downey Jr., Scarlett Johansson, Sebastian Stan, Anthony Mackie, Don Cheadle, Jeremy Renner, Chadwick Boseman, Paul Bettany, Elizabeth Olsen"
284054,"Chadwick Boseman, Michael B. Jordan, Lupita Nyong'o, Danai Gurira, Martin Freeman, Daniel Kaluuya, Letitia Wright, Winston Duke, Sterling K. Brown, Angela Bassett"
284052,"Benedict Cumberbatch, Chiwetel Ejiofor, Rachel McAdams, Benedict Wong, Mads Mikkelsen, Tilda Swinton, Michael Stuhlbarg, Benjamin Bratt, Scott Adkins, Zara Phythian"
315635,"Tom Holland, Michael Keaton, Robert Downey Jr., Marisa Tomei, Jon Favreau, Gwyneth Paltrow, Zendaya, Donald Glover, Jacob Batalon, Laura Harrier"
283995,"Chris Pratt, Zoe Saldaña, Dave Bautista, Vin Diesel, Bradley Cooper, Kurt Russell, Michael Rooker, Karen Gillan, Pom Klementieff, Sylvester Stallone,"
297761,"Will Smith, Jared Leto, Margot Robbie, Joel Kinnaman, Viola Davis, Jai Courtney, Jay Hernandez, Adewale Akinnuoye-Agbaje, Cara Delevingne, Ike Barinholtz"


In [0]:
display(
    df_credits_raw
    .select(
        "id",
        "directors",
        "writers"
    )
    .where(
        F.col("directors").isNotNull()
        |
        F.col("writers").isNotNull()
    )
    .limit(100)
)

id,directors,writers
293660,Tim Miller,"Rhett Reese, Paul Wernick"
299536,"Anthony Russo, Joe Russo",N/A
299534,"Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee, Jack Kirby, Joe Simon, Steve Englehart, Steve Gan, Bill Mantlo, Keith Giffen, Jim Starlin, Larry Lieber, Don Heck"
475557,Todd Phillips,"Todd Phillips, Scott Silver, Bob Kane, Bill Finger, Jerry Robinson"
271110,"Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Joe Simon, Jack Kirby"
284054,Ryan Coogler,"Ryan Coogler, Joe Robert Cole, Stan Lee, Jack Kirby"
284052,Scott Derrickson,N/A
315635,Jon Watts,"Jonathan Goldstein, John Francis Daley, Jon Watts, Christopher Ford, Chris McKenna, Erik Sommers, Stan Lee, Steve Ditko, Joe Simon, Jack Kirby"
283995,James Gunn,"James Gunn, Dan Abnett, Andy Lanning, Steve Englehart, Steve Gan, Jim Starlin, Stan Lee, Larry Lieber, Jack Kirby, Bill Mantlo, Keith Giffen, Steve Gerber, Val Mayerik"
297761,David Ayer,"David Ayer, John Ostrander"


In [0]:
display(
    df_credits_raw
    .select(
        "id",
        "production_companies"
    )
    .where(
        F.col("production_companies").isNotNull()
    )
    .limit(100)
)

id,production_companies
293660,"20th Century Fox, The Donners' Company, Genre Films"
299536,Marvel Studios
299534,Marvel Studios
475557,"Warner Bros. Pictures, Joint Effort, Village Roadshow Pictures, Bron Studios, DC Films"
271110,Marvel Studios
284054,Marvel Studios
284052,Marvel Studios
315635,"Marvel Studios, Pascal Pictures, LStar Capital, Columbia Pictures"
283995,Marvel Studios
297761,"DC Entertainment, Dune Entertainment, Warner Bros. Pictures, Atlas Entertainment, DC Comics, DC Films, Lin Pictures"


In [0]:
display(
    df_credits_raw
    .select(
        "id",
        "genres"
    )
    .filter(
        F.col("genres").rlike(r"(^|[,;]\s*)[-+]?[0-9]+(?:\.[0-9]+)?(\s*[,;]|$)")
    )
    .limit(100)
)

id,genres
491283,11.491
354556,15.591
542830,14.51
588001,8.465
486103,7.909
614587,13.978
589984,6.894
528773,5.623
375900,6.353
480330,5.709


In [0]:
display(
    df_credits_raw
    .select(
        "id",
        "genres",
        F.length("genres").alias("tamanho")
    )
    .where(F.col("genres").isNotNull())
    .orderBy(
        F.col("tamanho").desc()
    )
    .limit(100)
)

id,genres,tamanho
485580,hospice care and home. Diane is a nurse caring for end-stage cancer patients when she is diagnosed with ovarian cancer herself. 23-year-old Alena undergoes a risky brain surgery that destroys her short-term memory. 95-year-old Berthold lives with his elderly wife who struggles to honor his wish of dying peacefully at home. Defining Hope follows these patients and others- and the nurses that guide them along the way- as they face death,438
720711,"Mal 3:2) The prophet Isaiah asked \""""\""""Who among us shall dwell with the devouring fire? who among us shall dwell with everlasting burnings?\""""\"""" (Isa 33:14) And then Isaiah answers the question and says \""""\""""he that walketh righteously.\""""\"""" (Isa. 33:15) Only the righteous may stand in the presence of Christ when He comes as a consuming fire and the message of Noah will prepare a people for that event""""""",412
512218,"she found herself unable to lose the muscle she so desperately gained. She now finds herself living one day as an alpha male and the next day as a delicate girl. Will Janae be able to handle her muscle relapses? Will her passage from being a male bring her the peace she's looking for? Will society accept a 250lbs muscular woman? Is her path personal redemption or physical and psychological disaster?""",403
664302,"""\"""" has become the mascot to a vast online community consisting of self-described \""""\""""hyper-anonymous twenty somethings\""""\"""" and \""""\""""guys who slipped between the cracks.\""""\"""" TFW No GF asks: How has the zeitgeist come to bear down on a generation alienated by the 'real world'? Meet the lost boys who came of age on the internet- places like 4chan and Twitter""""""",370
587272,you already consider that overflowing woman drowns in a glass of water. Javier will have to face the reality of dealing with five children (between four and twelve years old) when his wife decides to go on a trip and leave him alone with them. The chaotic situation that takes place at home will progressively evolve ecologically to the most absolute disaster,359
648677,"have finally uncovered the whereabouts of the Psycho Surgeons and get set to exact the bloodiest of revenge. But what does this mean for The Gore Collector himself? As he returns to where it all started - the Bunker of Blood - he will find out what this splatter-soaked road trip across a fevered nightmarescape has REALLY done to his mind AND body!""",350
801485,"""builds a machine known as the \""""Resonator\""""\"""". The machine allows one to experience multiple dimensions while navigating the unsavory beasts that dwell within them. But things get complicated when Tillinghast realizes that the prototype of his creation has not only released murderous and deadly creatures into his world""""""",327
829830,"for his mother's surgery. Both of them become friends as Sabu's love story helps Kevin realize his mistakes in his marriage life.The movie revolves around the journey of Sabu's love life 10 years back and the journey of Kevin's realization. The movie also highlights other bystanders life story which can be related to""",320
464933,000 Catholic and Protestant children to suburban US for summer-long visits where they forged unexpected friendships and found they had more in common with the 'enemy' than they thought. Now this extraordinary untold story is being brought to the screen in a new documentary by Des Henderson,290
464933,000 Catholic and Protestant children to suburban US for summer-long visits where they forged unexpected friendships and found they had more in common with the 'enemy' than they thought. Now this extraordinary untold story is being brought to the screen in a new documentary by Des Henderson,290


In [0]:
GENRES_TARGET = (
    f"{CATALOG}.{SILVER_SCHEMA}.tb_generos"
)

PEOPLE_COMPANIES_TARGET = (
    f"{CATALOG}.{SILVER_SCHEMA}.tb_pessoas_empresas"
)

In [0]:
GENEROS_VALIDOS = [
    "Action",
    "Adventure",
    "Animation",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Family",
    "Fantasy",
    "History",
    "Horror",
    "Music",
    "Mystery",
    "Romance",
    "Science Fiction",
    "TV Movie",
    "Thriller",
    "War",
    "Western"
]

In [0]:
df_generos = (
    df_credits_raw
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.col("genres")
    )
    .withColumn(
        "genero",
        F.explode(
            F.split(
                F.regexp_replace(
                    F.col("genres"),
                    ";",
                    ","
                ),
                ","
            )
        )
    )
    .withColumn(
        "genero",
        F.trim(F.col("genero"))
    )
)

In [0]:
mapa_generos = F.create_map(
    *[
        item
        for genero in GENEROS_VALIDOS
        for item in (
            F.lit(genero.lower()),
            F.lit(genero)
        )
    ]
)

df_generos = (
    df_generos
    .withColumn(
        "genero_normalizado",
        F.lower(F.trim(F.col("genero")))
    )
    .withColumn(
        "genero",
        mapa_generos[F.col("genero_normalizado")]
    )
    .filter(
        F.col("genero").isNotNull()
    )
    .drop("genero_normalizado")
)

In [0]:
df_generos_final = (
    df_generos
    .select(
        "id_filme",
        "genero"
    )
    .dropDuplicates([
        "id_filme",
        "genero"
    ])
)

In [0]:
total_generos = df_generos_final.count()

ids_generos_nulos = (
    df_generos_final
    .filter(F.col("id_filme").isNull())
    .count()
)

generos_nulos = (
    df_generos_final
    .filter(
        F.col("genero").isNull()
        |
        (F.trim(F.col("genero")) == "")
    )
    .count()
)

duplicatas_generos = (
    df_generos_final
    .groupBy(
        "id_filme",
        "genero"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("=" * 60)
print("VALIDAÇÃO — TB_GENEROS")
print("=" * 60)

print(f"Total de relações filme/gênero: {total_generos}")
print(f"IDs nulos:                      {ids_generos_nulos}")
print(f"Gêneros nulos/vazios:           {generos_nulos}")
print(f"Duplicatas restantes:            {duplicatas_generos}")

if (
    ids_generos_nulos == 0
    and generos_nulos == 0
    and duplicatas_generos == 0
):
    print("[OK] Todas as regras de gêneros foram atendidas.")
else:
    print("[ERRO] Existem inconsistências em gêneros.")

VALIDAÇÃO — TB_GENEROS
Total de relações filme/gênero: 132775
IDs nulos:                      0
Gêneros nulos/vazios:           0
Duplicatas restantes:            0
[OK] Todas as regras de gêneros foram atendidas.


In [0]:
display(
    df_generos_final
    .groupBy("genero")
    .count()
    .orderBy(
        F.col("count").desc()
    )
)

genero,count
Drama,30834
Documentary,18860
Comedy,17570
Thriller,9547
Horror,9287
Romance,7075
Action,5598
Crime,4365
Animation,4214
TV Movie,3714


In [0]:
(
    df_generos_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GENRES_TARGET)
)

print(
    f"[OK] Tabela criada/atualizada: "
    f"{GENRES_TARGET}"
)

[OK] Tabela criada/atualizada: workspace.silver.tb_generos


In [0]:
df_generos_validacao = spark.table(
    GENRES_TARGET
)

print("=" * 60)
print("VALIDAÇÃO FINAL — SILVER.TB_GENEROS")
print("=" * 60)

print(f"Tabela: {GENRES_TARGET}")
print(f"Registros: {df_generos_validacao.count()}")

df_generos_validacao.printSchema()

display(
    df_generos_validacao.limit(20)
)

VALIDAÇÃO FINAL — SILVER.TB_GENEROS
Tabela: workspace.silver.tb_generos
Registros: 132775
root
 |-- id_filme: string (nullable = true)
 |-- genero: string (nullable = true)



id_filme,genero
269149,Animation
429617,Science Fiction
330459,Action
127380,Family
332562,Romance
291805,Action
293167,Action
381284,History
454626,Action
454626,Science Fiction


In [0]:
def extrair_entidades(df, coluna_origem, tipo):
    return (
        df
        .select(
            F.col("id").cast("string").alias("id_filme"),
            F.col(coluna_origem).alias("entidades")
        )
        .where(F.col("entidades").isNotNull())
        .withColumn(
            "nome",
            F.explode(
                F.split(
                    F.regexp_replace(
                        F.col("entidades"),
                        ";",
                        ","
                    ),
                    ","
                )
            )
        )
        .withColumn(
            "nome",
            F.trim(F.col("nome"))
        )
        .filter(
            F.col("nome").isNotNull()
            & (F.col("nome") != "")
        )
        .withColumn(
            "tipo",
            F.lit(tipo)
        )
        .select(
            "id_filme",
            "nome",
            "tipo"
        )
    )

In [0]:
df_atores = extrair_entidades(
    df_credits_raw,
    "cast",
    "Ator"
)

print(f"Relações com atores: {df_atores.count()}")

display(df_atores.limit(20))

Relações com atores: 585916


id_filme,nome,tipo
293660,Ryan Reynolds,Ator
293660,Morena Baccarin,Ator
293660,Ed Skrein,Ator
293660,T.J. Miller,Ator
293660,Gina Carano,Ator
293660,Leslie Uggams,Ator
293660,Brianna Hildebrand,Ator
293660,Stefan Kapičić,Ator
293660,Karan Soni,Ator
293660,Randal Reeder,Ator


In [0]:
df_diretores = extrair_entidades(
    df_credits_raw,
    "directors",
    "Diretor"
)

print(f"Relações com diretores: {df_diretores.count()}")

display(df_diretores.limit(20))

Relações com diretores: 115662


id_filme,nome,tipo
293660,Tim Miller,Diretor
299536,Anthony Russo,Diretor
299536,Joe Russo,Diretor
299534,Anthony Russo,Diretor
299534,Joe Russo,Diretor
475557,Todd Phillips,Diretor
271110,Anthony Russo,Diretor
271110,Joe Russo,Diretor
284054,Ryan Coogler,Diretor
284052,Scott Derrickson,Diretor


In [0]:
df_roteiristas = extrair_entidades(
    df_credits_raw,
    "writers",
    "Roteirista"
)

print(
    f"Relações com roteiristas: "
    f"{df_roteiristas.count()}"
)

display(df_roteiristas.limit(20))

Relações com roteiristas: 151506


id_filme,nome,tipo
293660,Rhett Reese,Roteirista
293660,Paul Wernick,Roteirista
299536,N/A,Roteirista
299534,Christopher Markus,Roteirista
299534,Stephen McFeely,Roteirista
299534,Stan Lee,Roteirista
299534,Jack Kirby,Roteirista
299534,Joe Simon,Roteirista
299534,Steve Englehart,Roteirista
299534,Steve Gan,Roteirista


In [0]:
df_produtoras = extrair_entidades(
    df_credits_raw,
    "production_companies",
    "Produtora"
)

print(
    f"Relações com produtoras: "
    f"{df_produtoras.count()}"
)

display(df_produtoras.limit(20))

Relações com produtoras: 126906


id_filme,nome,tipo
293660,20th Century Fox,Produtora
293660,The Donners' Company,Produtora
293660,Genre Films,Produtora
299536,Marvel Studios,Produtora
299534,Marvel Studios,Produtora
475557,Warner Bros. Pictures,Produtora
475557,Joint Effort,Produtora
475557,Village Roadshow Pictures,Produtora
475557,Bron Studios,Produtora
475557,DC Films,Produtora


In [0]:
df_pessoas_empresas = (
    df_atores
    .unionByName(df_diretores)
    .unionByName(df_roteiristas)
    .unionByName(df_produtoras)
)

display(
    df_pessoas_empresas.limit(50)
)

id_filme,nome,tipo
293660,Ryan Reynolds,Ator
293660,Morena Baccarin,Ator
293660,Ed Skrein,Ator
293660,T.J. Miller,Ator
293660,Gina Carano,Ator
293660,Leslie Uggams,Ator
293660,Brianna Hildebrand,Ator
293660,Stefan Kapičić,Ator
293660,Karan Soni,Ator
293660,Randal Reeder,Ator


In [0]:
df_pessoas_empresas = (
    df_pessoas_empresas
    .withColumn(
        "nome",
        F.trim(
            F.regexp_replace(
                F.col("nome"),
                r"\s+",
                " "
            )
        )
    )
)

In [0]:
df_pessoas_empresas = (
    df_pessoas_empresas
    .filter(
        ~F.col("nome").rlike(
            r"^[+-]?[0-9]+(?:[.,][0-9]+)?$"
        )
    )
    .filter(
        F.length(F.trim(F.col("nome"))) > 0
    )
)

In [0]:
df_pessoas_empresas_final = (
    df_pessoas_empresas
    .select(
        "id_filme",
        "nome",
        "tipo"
    )
    .dropDuplicates([
        "id_filme",
        "nome",
        "tipo"
    ])
)

In [0]:
display(
    df_pessoas_empresas_final
    .groupBy("tipo")
    .count()
    .orderBy(F.col("count").desc())
)

tipo,count
Ator,547302
Roteirista,138708
Produtora,120246
Diretor,109138


In [0]:
display(
    df_pessoas_empresas_final
    .select(
        "id_filme",
        "nome",
        "tipo",
        F.length("nome").alias("tamanho_nome")
    )
    .orderBy(
        F.col("tamanho_nome").desc()
    )
    .limit(100)
)

id_filme,nome,tipo,tamanho_nome
720153,"it's a microcosm of what we all need a little more time with - slowing things down and challenging yourself to accomplish something that once seemed impossible... like riding scooters across the country in 11 days. This adventurous and often hilarious film documents the story of eight Soldiers of Destiny Scooter Club members as they traverse from the white sand beaches of FL through some of the most beautiful - and sometimes most desolate - pockets of the United States.""",Produtora,475
393796,"but the amateur accolades leading to Olympic accomplishments were blown off the podiums in the 1952 Helsinki Games. Roger Bannister was the epitome of that disappearing scholar-athlete ideal. Can the lunchtime-trained runner immersed in his medical school studies inject the booster shot into Britain's flagging but still flickering morale?""",Produtora,341
1068607,"and Koyomi Yamanaka—an unemployed man who was wandering on the street—into its cockpit. Their encounter with the kaiju marks the beginning of their entanglement with kaiju eugenicists—kaiju users who manipulate kaiju with ill intent—and their efforts toward bringing out the full potential of Dynazenon.""",Diretor,304
773394,cry and find the true meaning and spirit of Christmas as they experience this never-before-seen edition of McLean's holiday classic. Audiences will be spellbound as Uncle John recounts the story of Christ's birth as told by lesser-known characters from the Nativity through story and song. The Innkeeper,Produtora,303
527490,"""borders. I don't understand it … I just want to see you!\"""" The newly discovered planet Emanon emerges one night at the sky. A weird customer brings Mika a strange present from his journey to \""""\""""Agartha\""""\"""". And she... is still unable to cope with the sudden parting of Teru.\""""""""""",Ator,287
765673,"""CBS aired the two-hour \""""Star Wars Holiday Special\""""\"""" during the week of Thanksgiving. The broadcast was watched by 13 million people. It never re-aired and is considered one of the worst shows to ever air on TV. While some fans of the franchise are aware of this dark secret""""""",Produtora,283
535102,"is able to unravel the most mysterious thing. The detective immediately established a circle of suspects and began to screen out those who had an iron alibi. But suddenly his list began to decline for a completely different reason - someone began to kill the suspects one by one...""",Produtora,282
759271,California. The film reveals the dramatic decline of the American interior through a combination of emotional personal stories and thoughtful conservative commentary. Filmmaker Christopher F. Rufo spent five years gathering these intimate portraits of Americans on the edge,Produtora,273
590658,"""and constant systemic racism. Artist \""""Just Chase\""""\"""" paints a picture of his life in crime and the events that made him get out out of the street life to chase his musical dreams. \""""\""""This is North Preston\""""\"""" gains insight from respected community members""""""",Produtora,268
498509,"""and self-expression a form of communication? Mattioli Production's upcoming documentary \""""Words\""""\"""" will explore how people navigate gender and identity in the open and evolving landscape of New York City. Using some if NYC's most fluid scenes as a backdrop""""""",Produtora,264


In [0]:
total_entidades = (
    df_pessoas_empresas_final.count()
)

ids_nulos = (
    df_pessoas_empresas_final
    .filter(F.col("id_filme").isNull())
    .count()
)

nomes_invalidos = (
    df_pessoas_empresas_final
    .filter(
        F.col("nome").isNull()
        |
        (F.trim(F.col("nome")) == "")
    )
    .count()
)

tipos_invalidos = (
    df_pessoas_empresas_final
    .filter(
        ~F.col("tipo").isin(
            "Ator",
            "Diretor",
            "Roteirista",
            "Produtora"
        )
    )
    .count()
)

duplicatas = (
    df_pessoas_empresas_final
    .groupBy(
        "id_filme",
        "nome",
        "tipo"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("=" * 60)
print("VALIDAÇÃO — TB_PESSOAS_EMPRESAS")
print("=" * 60)

print(f"Total de relações:       {total_entidades}")
print(f"IDs nulos:               {ids_nulos}")
print(f"Nomes nulos/vazios:      {nomes_invalidos}")
print(f"Tipos inválidos:          {tipos_invalidos}")
print(f"Duplicatas restantes:     {duplicatas}")

if (
    ids_nulos == 0
    and nomes_invalidos == 0
    and tipos_invalidos == 0
    and duplicatas == 0
):
    print(
        "[OK] Validação estrutural concluída."
    )
else:
    print(
        "[ERRO] Existem inconsistências."
    )

VALIDAÇÃO — TB_PESSOAS_EMPRESAS
Total de relações:       915394
IDs nulos:               0
Nomes nulos/vazios:      0
Tipos inválidos:          0
Duplicatas restantes:     0
[OK] Validação estrutural concluída.


In [0]:
df_tamanho_entidades = (
    df_pessoas_empresas_final
    .select(
        F.length("nome").alias("tamanho_nome")
    )
)

display(
    df_tamanho_entidades
    .select(
        F.expr(
            """
            percentile_approx(
                tamanho_nome,
                array(0.50, 0.90, 0.95, 0.99, 0.995, 0.999)
            )
            """
        ).alias("percentis")
    )
)

percentis
"List(13, 19, 21, 28, 32, 48)"


In [0]:
display(
    df_pessoas_empresas_final
    .withColumn(
        "faixa_tamanho",
        F.when(F.length("nome") <= 50, "01 - até 50")
         .when(F.length("nome") <= 75, "02 - 51 a 75")
         .when(F.length("nome") <= 100, "03 - 76 a 100")
         .when(F.length("nome") <= 150, "04 - 101 a 150")
         .otherwise("05 - acima de 150")
    )
    .groupBy(
        "tipo",
        "faixa_tamanho"
    )
    .count()
    .orderBy(
        "tipo",
        "faixa_tamanho"
    )
)

tipo,faixa_tamanho,count
Ator,01 - até 50,547285
Ator,02 - 51 a 75,4
Ator,03 - 76 a 100,8
Ator,04 - 101 a 150,4
Ator,05 - acima de 150,1
Diretor,01 - até 50,109043
Diretor,02 - 51 a 75,34
Diretor,03 - 76 a 100,26
Diretor,04 - 101 a 150,23
Diretor,05 - acima de 150,12


In [0]:
display(
    df_pessoas_empresas_final
    .filter(
        F.length("nome").between(60, 150)
    )
    .select(
        "id_filme",
        "nome",
        "tipo",
        F.length("nome").alias("tamanho_nome")
    )
    .orderBy(
        F.col("tamanho_nome").desc()
    )
    .limit(200)
)

id_filme,nome,tipo,tamanho_nome
481745,a development town in the Negev desert. Their personal stories recount of the price immigrant-families pay and the price still paid by Israeli society,Produtora,150
576357,"decides to break his scientific experiments and win the heart of Paradise as she unfolds the mystery of a mysterious werewolf wandering in the woods.""",Produtora,150
944041,and caring brought long-lost love and warmth to Zhao Yimei. The two became increasingly harmonious and created many warm and happy memories. In return,Produtora,150
682927,we are taken into a unique cinematic soundscape that doubles as a love letter to radiophonic art and its disarming insight into what makes us remember,Roteirista,150
390923,"so the drivers and supporting staff will be fired. But it is way more than that. Without the old trolleybuses the city is never going to be the same.""",Produtora,150
718350,"but Kyabla finds Satkari himself is very misleading person. They try to reveal the mystery behind Satkari's story. Will they survive from the danger?""",Produtora,150
1012763,"Martin is bewildered at Jonah's rejection of his love and must come to terms with the uniqueness of his child and perhaps his own emotional needs.""",Roteirista,147
428684,"plus lead exhibit designer Patrick Marsh and lead construction engineer LeRoy Troyer. Share the excitement! English subtitles. Approx. 30 minutes.""",Diretor,147
489428,"Ted follows a hot vampire into group therapy where the patients all suffer disorders of the paranormal. Yep. Things just went from weird to wacky.""",Produtora,147
566057,"where they run into the Kyuranger team as they pass through space. Just who exactly kidnapped them? And why did the 12 Kyurangers return to space?""",Produtora,147


In [0]:
LIMITE_TAMANHO_ENTIDADE = 60

df_pessoas_empresas_limpo = (
    df_pessoas_empresas_final
    .filter(
        F.length(F.col("nome"))
        <= LIMITE_TAMANHO_ENTIDADE
    )
)

In [0]:
total_antes = df_pessoas_empresas_final.count()
total_depois = df_pessoas_empresas_limpo.count()

removidos = total_antes - total_depois

print("=" * 60)
print("LIMPEZA — COLUMN SHIFT")
print("=" * 60)

print(f"Registros antes:   {total_antes}")
print(f"Registros depois:  {total_depois}")
print(f"Registros removidos: {removidos}")

print(
    f"Percentual removido: "
    f"{(removidos / total_antes) * 100:.4f}%"
)

LIMPEZA — COLUMN SHIFT
Registros antes:   915394
Registros depois:  914823
Registros removidos: 571
Percentual removido: 0.0624%


In [0]:
display(
    df_pessoas_empresas_limpo
    .select(
        "id_filme",
        "nome",
        "tipo",
        F.length("nome").alias("tamanho_nome")
    )
    .orderBy(
        F.col("tamanho_nome").desc()
    )
    .limit(100)
)

id_filme,nome,tipo,tamanho_nome
833510,"it is not always possible to remain an unaffected observer.""",Produtora,60
1328049,leading to a dramatic showdown and a heartwarming conclusion,Diretor,60
602519,A letter arrived from our mother who left us many years ago.,Produtora,60
1218435,National Association of Latino Independent Producers (NALIP),Produtora,60
624417,Traces the long and ferocious battle between Cola and Pepsi.,Diretor,60
374452,Studioul de Creatie Cinematografica al Ministerului Culturii,Produtora,60
871882,"""who dreams of realizing his \""""Ukrainian dream\""""\"""".\""""""""""",Produtora,60
771199,Hrvatski centar za istraživačko novinarstvo i slobodu medija,Roteirista,60
1211221,National Association of Latino Independent Producers (NALIP),Produtora,60
1172437,National Association of Latino Independent Producers (NALIP),Produtora,60


In [0]:
total_entidades = (
    df_pessoas_empresas_limpo.count()
)

ids_nulos = (
    df_pessoas_empresas_limpo
    .filter(F.col("id_filme").isNull())
    .count()
)

nomes_invalidos = (
    df_pessoas_empresas_limpo
    .filter(
        F.col("nome").isNull()
        |
        (F.trim(F.col("nome")) == "")
    )
    .count()
)

tipos_invalidos = (
    df_pessoas_empresas_limpo
    .filter(
        ~F.col("tipo").isin(
            "Ator",
            "Diretor",
            "Roteirista",
            "Produtora"
        )
    )
    .count()
)

duplicatas = (
    df_pessoas_empresas_limpo
    .groupBy(
        "id_filme",
        "nome",
        "tipo"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

residuos_longos = (
    df_pessoas_empresas_limpo
    .filter(
        F.length("nome")
        > LIMITE_TAMANHO_ENTIDADE
    )
    .count()
)

print("=" * 60)
print("VALIDAÇÃO FINAL — TB_PESSOAS_EMPRESAS")
print("=" * 60)

print(f"Total de relações:       {total_entidades}")
print(f"IDs nulos:               {ids_nulos}")
print(f"Nomes nulos/vazios:      {nomes_invalidos}")
print(f"Tipos inválidos:          {tipos_invalidos}")
print(f"Duplicatas restantes:     {duplicatas}")
print(f"Resíduos longos:          {residuos_longos}")

if (
    ids_nulos == 0
    and nomes_invalidos == 0
    and tipos_invalidos == 0
    and duplicatas == 0
    and residuos_longos == 0
):
    print(
        "[OK] Todas as regras de pessoas/empresas "
        "foram atendidas."
    )
else:
    print("[ERRO] Existem inconsistências.")

VALIDAÇÃO FINAL — TB_PESSOAS_EMPRESAS
Total de relações:       914823
IDs nulos:               0
Nomes nulos/vazios:      0
Tipos inválidos:          0
Duplicatas restantes:     0
Resíduos longos:          0
[OK] Todas as regras de pessoas/empresas foram atendidas.


In [0]:
display(
    df_pessoas_empresas_limpo
    .groupBy("tipo")
    .count()
    .orderBy(
        F.col("count").desc()
    )
)

tipo,count
Ator,547286
Roteirista,138661
Produtora,119819
Diretor,109057


In [0]:
(
    df_pessoas_empresas_limpo.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PEOPLE_COMPANIES_TARGET)
)

print(
    f"[OK] Tabela criada/atualizada: "
    f"{PEOPLE_COMPANIES_TARGET}"
)

[OK] Tabela criada/atualizada: workspace.silver.tb_pessoas_empresas


In [0]:
df_pessoas_empresas_validacao = spark.table(
    PEOPLE_COMPANIES_TARGET
)

print("=" * 60)
print(
    "VALIDAÇÃO — "
    "SILVER.TB_PESSOAS_EMPRESAS"
)
print("=" * 60)

print(
    f"Tabela: {PEOPLE_COMPANIES_TARGET}"
)

print(
    f"Registros: "
    f"{df_pessoas_empresas_validacao.count()}"
)

df_pessoas_empresas_validacao.printSchema()

display(
    df_pessoas_empresas_validacao.limit(20)
)

VALIDAÇÃO — SILVER.TB_PESSOAS_EMPRESAS
Tabela: workspace.silver.tb_pessoas_empresas
Registros: 914823
root
 |-- id_filme: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- tipo: string (nullable = true)



id_filme,nome,tipo
299536,Don Cheadle,Ator
263115,Hugh Jackman,Ator
354912,Anthony Gonzalez,Ator
419430,Daniel Kaluuya,Ator
424694,Mike Myers,Ator
374720,Mark Rylance,Ator
335983,Tom Hardy,Ator
363088,Evangeline Lilly,Ator
363088,Randall Park,Ator
274870,Kara Flowers,Ator
